In [49]:
import pandas as pd
from pathlib import Path
from os import getcwd
import numpy as np

In [29]:
ROOT_DIR = Path(getcwd()).parent.parent
RAW_DIR = ROOT_DIR / "data/raw"
INTERIM_DIR = ROOT_DIR / "data/interim"
PROCESSED_DIR = ROOT_DIR / "data/processed"

# 1. SP100 Data

In [ ]:
df = pd.read_parquet(RAW_DIR / "prices_raw.parquet")
print(df.head())

..\..\data\raw
        date ticker   adj_close       close        high         low  \
0 2021-01-04   AAPL  125.856697  129.410004  133.610001  126.760002   
1 2021-01-05   AAPL  127.412758  131.009995  131.740005  128.429993   
2 2021-01-06   AAPL  123.123840  126.599998  131.050003  126.379997   
3 2021-01-07   AAPL  127.325241  130.919998  131.630005  127.860001   
4 2021-01-08   AAPL  128.424240  132.050003  132.630005  130.229996   

         open     volume  
0  133.520004  143301900  
1  128.889999   97664900  
2  127.720001  155088000  
3  128.360001  109578200  
4  132.429993  105158200  


# 2. Metadata Exploration

In [4]:
metadata = pd.read_parquet(RAW_DIR / "ticker_metadata.parquet")
print(metadata.head())

  ticker      sector                         industry     market_cap  \
0   AAPL  Technology             Consumer Electronics  3776182222848   
1   ABBV  Healthcare     Drug Manufacturers - General   360473788416   
2    ABT  Healthcare                  Medical Devices   172512804864   
3    ACN  Technology  Information Technology Services   114569404416   
4   ADBE  Technology           Software - Application    96137871360   

            short_name currency  
0           Apple Inc.      USD  
1          AbbVie Inc.      USD  
2  Abbott Laboratories      USD  
3        Accenture plc      USD  
4           Adobe Inc.      USD  


In [8]:
# What columns are in the metadata?
print(metadata.columns)

Index(['ticker', 'sector', 'industry', 'market_cap', 'short_name', 'currency'], dtype='str')


In [11]:
# Unique sectors and number of tickers in each sector
sector_counts = metadata["sector"].value_counts()
print(sector_counts)
print(f"The number of sectors is: {metadata['sector'].nunique()}")

sector
Financial Services        16
Healthcare                14
Technology                12
Industrials               12
Consumer Defensive        10
Consumer Cyclical          9
Communication Services     7
Energy                     3
Utilities                  3
Real Estate                2
Name: count, dtype: int64
The number of sectors is: 10


In [7]:
# Unique industries and number of tickers in each industry
industry_counts = metadata["industry"].value_counts()
print(industry_counts)

industry
Drug Manufacturers - General           8
Credit Services                        5
Aerospace & Defense                    5
Banks - Diversified                    5
Semiconductors                         4
Telecom Services                       3
Discount Stores                        3
Utilities - Regulated Electric         3
Medical Devices                        2
Information Technology Services        2
Software - Application                 2
Household & Personal Products          2
Healthcare Plans                       2
Oil & Gas Integrated                   2
Diagnostics & Research                 2
Entertainment                          2
Integrated Freight & Logistics         2
Auto Manufacturers                     2
Internet Content & Information         2
Capital Markets                        2
Home Improvement Retail                2
Conglomerates                          2
Beverages - Non-Alcoholic              2
Restaurants                            2
Tobacco

In [10]:
# Are all tickers traded using the same currency?
currency_counts = metadata["currency"].value_counts()
print(currency_counts)

currency
USD    88
Name: count, dtype: int64


# 3. Base Panel

In [24]:
interim = pd.read_parquet(INTERIM_DIR / "base_panel.parquet")
print(interim.head())

        date ticker   adj_close       close        high         low  \
0 2021-01-04   AAPL  125.856697  129.410004  133.610001  126.760002   
1 2021-01-05   AAPL  127.412758  131.009995  131.740005  128.429993   
2 2021-01-06   AAPL  123.123840  126.599998  131.050003  126.379997   
3 2021-01-07   AAPL  127.325241  130.919998  131.630005  127.860001   
4 2021-01-08   AAPL  128.424240  132.050003  132.630005  130.229996   

         open     volume      sector              industry     market_cap  \
0  133.520004  143301900  Technology  Consumer Electronics  3776182222848   
1  128.889999   97664900  Technology  Consumer Electronics  3776182222848   
2  127.720001  155088000  Technology  Consumer Electronics  3776182222848   
3  128.360001  109578200  Technology  Consumer Electronics  3776182222848   
4  132.429993  105158200  Technology  Consumer Electronics  3776182222848   

   short_name currency  
0  Apple Inc.      USD  
1  Apple Inc.      USD  
2  Apple Inc.      USD  
3  Apple I

# 4. Features

In [25]:
features = pd.read_parquet(INTERIM_DIR / "features_panel.parquet")
print(features.head())

        date ticker   adj_close       close        high         low  \
0 2021-01-04   AAPL  125.856697  129.410004  133.610001  126.760002   
1 2021-01-05   AAPL  127.412758  131.009995  131.740005  128.429993   
2 2021-01-06   AAPL  123.123840  126.599998  131.050003  126.379997   
3 2021-01-07   AAPL  127.325241  130.919998  131.630005  127.860001   
4 2021-01-08   AAPL  128.424240  132.050003  132.630005  130.229996   

         open     volume      sector              industry  ...  short_name  \
0  133.520004  143301900  Technology  Consumer Electronics  ...  Apple Inc.   
1  128.889999   97664900  Technology  Consumer Electronics  ...  Apple Inc.   
2  127.720001  155088000  Technology  Consumer Electronics  ...  Apple Inc.   
3  128.360001  109578200  Technology  Consumer Electronics  ...  Apple Inc.   
4  132.429993  105158200  Technology  Consumer Electronics  ...  Apple Inc.   

  currency log_ret_1d  adj_close_ret_1d     ma_5  ma_20  volume_ma_20  \
0      USD        NaN    

In [27]:
print(features["ticker"].nunique())

88


In [26]:
# What additional features were added which were not in the base panel?
base_columns = set(interim.columns)
feature_columns = set(features.columns)
new_features = feature_columns - base_columns
print(f"New features added: {new_features}")

New features added: {'rsi_14', 'roll_vol_20', 'volume_norm', 'log_ret_1d', 'ma_20', 'adj_close_ret_1d', 'ma_5', 'volume_ma_20'}


# 5. Targets Computation

In [35]:
targets = pd.read_parquet(PROCESSED_DIR / "panel_with_targets.parquet")
print(targets.head())

        date ticker   adj_close       close        high         low  \
0 2021-01-04   AAPL  125.856697  129.410004  133.610001  126.760002   
1 2021-01-05   AAPL  127.412758  131.009995  131.740005  128.429993   
2 2021-01-06   AAPL  123.123840  126.599998  131.050003  126.379997   
3 2021-01-07   AAPL  127.325241  130.919998  131.630005  127.860001   
4 2021-01-08   AAPL  128.424240  132.050003  132.630005  130.229996   

         open     volume      sector              industry  ...  log_ret_1d  \
0  133.520004  143301900  Technology  Consumer Electronics  ...         NaN   
1  128.889999   97664900  Technology  Consumer Electronics  ...    0.012288   
2  127.720001  155088000  Technology  Consumer Electronics  ...   -0.034241   
3  128.360001  109578200  Technology  Consumer Electronics  ...    0.033554   
4  132.429993  105158200  Technology  Consumer Electronics  ...    0.008594   

  adj_close_ret_1d     ma_5  ma_20  volume_ma_20  volume_norm  roll_vol_20  \
0              NaN  

In [36]:
print(f"Base panel columns: {interim.columns}")
print(f"Features panel columns: {features.columns}")
print(f"Targets panel columns: {targets.columns}")

Base panel columns: Index(['date', 'ticker', 'adj_close', 'close', 'high', 'low', 'open', 'volume',
       'sector', 'industry', 'market_cap', 'short_name', 'currency'],
      dtype='str')
Features panel columns: Index(['date', 'ticker', 'adj_close', 'close', 'high', 'low', 'open', 'volume',
       'sector', 'industry', 'market_cap', 'short_name', 'currency',
       'log_ret_1d', 'adj_close_ret_1d', 'ma_5', 'ma_20', 'volume_ma_20',
       'volume_norm', 'roll_vol_20', 'rsi_14'],
      dtype='str')
Targets panel columns: Index(['date', 'ticker', 'adj_close', 'close', 'high', 'low', 'open', 'volume',
       'sector', 'industry', 'market_cap', 'short_name', 'currency',
       'log_ret_1d', 'adj_close_ret_1d', 'ma_5', 'ma_20', 'volume_ma_20',
       'volume_norm', 'roll_vol_20', 'rsi_14', 'future_log_ret_5d',
       'target_class'],
      dtype='str')


In [37]:
# What new columns were added in each change?
features_added = feature_columns - base_columns
targets_added = set(targets.columns) - feature_columns
print(f"Features added: {features_added}")
print(f"Targets added: {targets_added}")

Features added: {'rsi_14', 'roll_vol_20', 'volume_norm', 'log_ret_1d', 'ma_20', 'adj_close_ret_1d', 'ma_5', 'volume_ma_20'}
Targets added: {'target_class', 'future_log_ret_5d'}


In [38]:
# Did any original columns get dropped in the process?
columns_dropped_in_features = base_columns - feature_columns
columns_dropped_in_targets = feature_columns - set(targets.columns)
print(f"Columns dropped in features: {columns_dropped_in_features}")
print(f"Columns dropped in targets: {columns_dropped_in_targets}")

Columns dropped in features: set()
Columns dropped in targets: set()


In [41]:
# Target distribution
print(targets["target_class"].value_counts().sort_index())

# Percentage distribution of target classes
target_distribution = targets["target_class"].value_counts(normalize=True).sort_index() * 100
print(target_distribution)

target_class
-1.0    37212
 0.0    27317
 1.0    45383
Name: count, dtype: int64
target_class
-1.0    33.856176
 0.0    24.853519
 1.0    41.290305
Name: proportion, dtype: float64


# 6. Splits Creation

In [42]:
data_with_splits = pd.read_parquet(PROCESSED_DIR / "panel_with_splits.parquet")
print(data_with_splits.head())

        date ticker   adj_close       close        high         low  \
0 2021-01-04   AAPL  125.856697  129.410004  133.610001  126.760002   
1 2021-01-05   AAPL  127.412758  131.009995  131.740005  128.429993   
2 2021-01-06   AAPL  123.123840  126.599998  131.050003  126.379997   
3 2021-01-07   AAPL  127.325241  130.919998  131.630005  127.860001   
4 2021-01-08   AAPL  128.424240  132.050003  132.630005  130.229996   

         open     volume      sector              industry  ...  \
0  133.520004  143301900  Technology  Consumer Electronics  ...   
1  128.889999   97664900  Technology  Consumer Electronics  ...   
2  127.720001  155088000  Technology  Consumer Electronics  ...   
3  128.360001  109578200  Technology  Consumer Electronics  ...   
4  132.429993  105158200  Technology  Consumer Electronics  ...   

   adj_close_ret_1d     ma_5 ma_20  volume_ma_20  volume_norm  roll_vol_20  \
0               NaN      NaN   NaN           NaN          NaN          NaN   
1          0.0

In [ ]:
# Class counts in each split
split_class_counts = data_with_splits.groupby("split")["target_class"].value_counts().unstack(fill_value=0)
print(split_class_counts)

target_class   -1.0    0.0    1.0
split                            
test           7159   5006   9307
train         22964  16301  26999
val            7089   6010   9077


In [44]:
# Percentages
split_class_percentages = split_class_counts.div(split_class_counts.sum(axis=1), axis=0) * 100
print(split_class_percentages)

target_class       -1.0        0.0        1.0
split                                        
test          33.341095  23.314083  43.344821
train         34.655318  24.600085  40.744597
val           31.966991  27.101371  40.931638


# 7. Tabular Dataset Creation

In [45]:
tabular_dataset = pd.read_parquet(PROCESSED_DIR / "tabular_dataset.parquet")
print(tabular_dataset.head())

  ticker       date  split  target_class      sector     market_cap  \
0   AAPL 2021-03-02  train          -1.0  Technology  3776182222848   
1   AAPL 2021-03-03  train          -1.0  Technology  3776182222848   
2   AAPL 2021-03-04  train           1.0  Technology  3776182222848   
3   AAPL 2021-03-05  train           0.0  Technology  3776182222848   
4   AAPL 2021-03-08  train           1.0  Technology  3776182222848   

   adj_close_mean_20  adj_close_std_20  adj_close_last  adj_close_min_20  ...  \
0         127.460158          5.189738      121.866333        117.843719  ...   
1         126.840277          5.443245      118.885887        117.843719  ...   
2         126.177467          5.800020      117.006081        117.006081  ...   
3         125.409715          5.779573      118.262550        117.006081  ...   
4         124.416241          6.069775      113.334129        113.334129  ...   

   rsi_14_mean_20  rsi_14_std_20  rsi_14_last  rsi_14_min_20  rsi_14_max_20  \
0      

# 8. Temporal Dataset Creation

In [50]:
temporal_dataset_meta = pd.read_parquet(PROCESSED_DIR / "temporal_index.parquet")
print(temporal_dataset_meta.head())

# We have both the X and y for the temporal dataset in npy files
X_temporal = np.load(PROCESSED_DIR / "X_temporal.npy")
y_temporal = np.load(PROCESSED_DIR / "y_temporal.npy")
print(f"X_temporal shape: {X_temporal.shape}")
print(f"y_temporal shape: {y_temporal.shape}")

   sample_id ticker       date  split  target_class
0          0   AAPL 2021-03-02  train            -1
1          1   AAPL 2021-03-03  train            -1
2          2   AAPL 2021-03-04  train             1
3          3   AAPL 2021-03-05  train             0
4          4   AAPL 2021-03-08  train             1
X_temporal shape: (106438, 20, 7)
y_temporal shape: (106438,)


# 9. Graph Construction

In [51]:
ticker_to_node_id = pd.read_parquet(PROCESSED_DIR / "ticker_to_node.parquet")
corr_edges = pd.read_parquet(PROCESSED_DIR / "graph_corr_edges.parquet")
js_edges = pd.read_parquet(PROCESSED_DIR / "graph_div_edges.parquet")

print(ticker_to_node_id.head())
print(corr_edges.head())
print(js_edges.head())

  ticker  node_id
0   AAPL        0
1   ABBV        1
2    ABT        2
3    ACN        3
4   ADBE        4
   src  dst src_ticker dst_ticker    weight    edge_type
0    0   58       AAPL       MSFT  0.722713  correlation
1    0   37       AAPL      GOOGL  0.665588  correlation
2    0   36       AAPL       GOOG  0.661749  correlation
3    0    4       AAPL       ADBE  0.635509  correlation
4    0   62       AAPL       NVDA  0.634759  correlation
   src  dst src_ticker dst_ticker    weight  distance      edge_type
0    0   76       AAPL        TMO  0.979376  0.021059  js_divergence
1    0   58       AAPL       MSFT  0.970300  0.030609  js_divergence
2    0   27       AAPL        DHR  0.968094  0.032958  js_divergence
3    0    3       AAPL        ACN  0.958334  0.043478  js_divergence
4    0   26       AAPL        CVX  0.957574  0.044306  js_divergence


# 10. GNN Dataset Creation

In [52]:
snapshot_dataset = pd.read_parquet(PROCESSED_DIR / "gnn_snapshots_index.parquet")
print(snapshot_dataset.head())

X = np.load(PROCESSED_DIR / "X_gnn.npy")
y = np.load(PROCESSED_DIR / "y_gnn.npy")

print(f"X_gnn shape: {X.shape}")
print(f"y_gnn shape: {y.shape}")

   snapshot_id       date  split  num_nodes
0           20 2021-03-02  train         88
1           21 2021-03-03  train         88
2           22 2021-03-04  train         88
3           23 2021-03-05  train         88
4           24 2021-03-08  train         88
X_gnn shape: (1168, 88, 20, 7)
y_gnn shape: (1168, 88)
